# 表格连接拼接与比较

学习目标：按键或标签组合小表，检查连接关系、未匹配项和结果行数，并比较两份同结构表格的差异。

前置知识：DataFrame、表格键、索引、缺失值、布尔筛选、分组与 MultiIndex 的基本含义。

运行环境：Python 3.12、pandas 3.0；反连接示例使用 pandas 3.0 新增的连接方式。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制小表，后续单元沿用首次导入的 pd。当前环境的默认 str 列使用 PyArrow 存储；本章不读写外部文件。

## 1 为订单补充客户信息

订单表保存客户编号，客户表保存地区。merge 按共同的 customer 键匹配，把地区补到每条订单上；how="left" 保留左侧订单。

下面客户表的编号唯一，同一客户可以有多条订单。连接按编号匹配，不要求两张表的行顺序一致。

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "order": ["O1", "O2", "O3"],
    "customer": ["C2", "C1", "C2"],
    "quantity": [2, 1, 3],
}, index=["row-A", "row-B", "row-C"])
customers = pd.DataFrame({"customer": ["C1", "C2"], "region": ["东区", "西区"]})
enriched = orders.merge(customers, on="customer", how="left", sort=False)

print(enriched)  # 三条订单依次补上西区、东区、西区。
print(enriched["order"].tolist(), enriched.shape)  # ['O1', 'O2', 'O3']，(3, 4)。
print(enriched.dtypes)  # quantity 为 int64，其余列为 str。
print(enriched.index.tolist())  # [0, 1, 2]：列对列连接不保留原行标签。

  order customer  quantity region
0    O1       C2         2     西区
1    O2       C1         1     东区
2    O3       C2         3     西区
['O1', 'O2', 'O3'] (3, 4)
order         str
customer      str
quantity    int64
region        str
dtype: object
[0, 1, 2]


## 2 选择保留的键

how 决定哪些键进入结果；它不保证每个键只产生一行。下面先使用两侧都唯一的键，只观察连接范围。

| how | 中文名称／含义 |
| --- | --- |
| inner | 内连接，只保留两侧都有的键；merge 的默认方式 |
| left | 左连接，保留左表的键 |
| right | 右连接，保留右表的键 |
| outer | 全外连接，保留两侧键的并集 |

sort=False 时，inner 和 left 保留左侧键顺序，right 保留右侧键顺序；outer 仍按键排序。没有匹配记录的一侧补缺失，数值列的 dtype 也可能因此变化。

In [2]:
left = pd.DataFrame({"key": ["B", "A"], "left_value": [20, 10]})
right = pd.DataFrame({"key": ["A", "C"], "right_value": [100, 300]})

for how in ("inner", "left", "right", "outer"):
    result = left.merge(right, on="key", how=how, sort=False, validate="one_to_one")
    print(how, result["key"].tolist(), result.shape)
    print(result)
# 键顺序分别为 [A]、[B, A]、[A, C]、[A, B, C]。
# 行数分别为 1、2、2、3；有缺失的普通整数列会转为 float64。
# validate 在这里确认两侧键唯一，具体关系在后面展开。

inner ['A'] (1, 3)
  key  left_value  right_value
0   A          10          100
left ['B', 'A'] (2, 3)
  key  left_value  right_value
0   B          20          NaN
1   A          10        100.0
right ['A', 'C'] (2, 3)
  key  left_value  right_value
0   A        10.0          100
1   C         NaN          300
outer ['A', 'B', 'C'] (3, 3)
  key  left_value  right_value
0   A        10.0        100.0
1   B        20.0          NaN
2   C         NaN        300.0


## 3 区分未匹配与数据本身缺失

indicator=True 添加 _merge 列，标明记录来自左侧、右侧还是两侧匹配，dtype 为 category。也可以传入列名字符串，给来源列命名。

不能只看地区是否缺失来判断客户是否匹配：客户表里的地区本来就可能缺失。下面 C2 已匹配但地区未填写，C3 才是未匹配客户。

In [3]:
orders = pd.DataFrame({"order": ["O1", "O2", "O3"], "customer": ["C1", "C2", "C3"]})
customers = pd.DataFrame({"customer": ["C1", "C2", "C4"], "region": ["东区", None, "北区"]})
audit = orders.merge(customers, on="customer", how="outer", indicator=True)

print(audit)
# C1、C2 为 both；C3 为 left_only；C4 为 right_only。
print(audit["_merge"].dtype)  # category。
print(audit.loc[audit["_merge"] == "left_only", ["order", "customer"]])
# 未匹配订单只有 O3；C2 的地区缺失不能作为未匹配证据。
print(audit.shape)  # (4, 4)。

  order customer region      _merge
0    O1       C1     东区        both
1    O2       C2    NaN        both
2    O3       C3    NaN   left_only
3   NaN       C4     北区  right_only
category
  order customer
2    O3       C3
(4, 4)


## 4 明确键与同名列

on 指定连接键；两侧键名不同时可以分别用 left_on、right_on。多个字段共同标识记录时，on 可以传列名列表。

未指定 on 时，merge 会把两表共有的列名都作为键。业务中应明确连接条件，避免把本来需要比较的同名列也当作键。非键同名列用 suffixes 区分来源，默认后缀为 _x、_y。

In [4]:
orders = pd.DataFrame({"order": ["O1", "O2"], "status": ["下单", "下单"]})
shipping = pd.DataFrame({"order": ["O2", "O1"], "status": ["待发", "已发"]})
result = orders.merge(
    shipping, on="order", how="left", suffixes=("_order", "_shipping"),
    validate="one_to_one", sort=False,
)

print(result)
# order 顺序为 O1、O2；订单状态都是下单，物流状态分别为已发、待发。
print(result.columns.tolist(), result.shape)
# ['order', 'status_order', 'status_shipping']，(2, 3)。
print(result.dtypes)  # 三列均为 str。

  order status_order status_shipping
0    O1           下单              已发
1    O2           下单              待发
['order', 'status_order', 'status_shipping'] (2, 3)
order              str
status_order       str
status_shipping    str
dtype: object


## 5 连接关系与 validate

连接基数描述一个键在左右表中能对应多少条记录。validate 检查指定一侧或两侧是否唯一；它检查键关系，不检查业务内容是否正确，也不检查是否全部匹配。

| validate | 中文名称／含义 |
| --- | --- |
| one_to_one | 一对一，左右键都必须唯一 |
| one_to_many | 一对多，左键必须唯一，右键允许重复 |
| many_to_one | 多对一，左键允许重复，右键必须唯一 |
| many_to_many | 多对多，允许两侧重复，不做唯一性检查 |

下面一张订单有多条商品明细，因此以订单表为左侧时使用 one_to_many。连接后每行表示一条明细，行数可以多于订单数。

In [5]:
orders = pd.DataFrame({"order": ["O1", "O2"], "customer": ["C1", "C2"]})
details = pd.DataFrame({
    "order": ["O1", "O1", "O2"], "item": ["笔", "本", "尺"], "quantity": [2, 1, 3],
})
joined = orders.merge(details, on="order", how="left", validate="one_to_many")

print(orders["order"].is_unique, details["order"].is_unique)  # True、False。
print(joined)  # O1 对应笔和本两行，O2 对应尺一行。
print(len(orders), len(details), len(joined), joined.shape)  # 2、3、3，(3, 4)。
print(joined["quantity"].sum())  # 6 件；这里每条明细恰好出现一次。

True False
  order customer item  quantity
0    O1       C1    笔         2
1    O1       C1    本         1
2    O2       C2    尺         3
2 3 3 (3, 4)
6


### 5.1 重复键的组合数量

同一个键在左侧出现 m 次、右侧出现 n 次时，匹配部分会产生 m × n 个组合，m、n 是该键两侧的记录数。多对多连接不会自动合并重复记录。

下面两个仓库记录都匹配三个供应商报价。六行是所有配对，不是六个独立的库存观测；汇总左表数量前需要重新判断统计单位。

下图只画同一个键 K：左侧两条、右侧三条记录两两匹配，得到六行。

![下图只画同一个键 K：左侧两条、右侧三条记录两两匹配，得到六行。](image/12-many-to-many.png)

In [6]:
stock = pd.DataFrame({"item": ["笔", "笔"], "warehouse": ["北仓", "南仓"], "quantity": [2, 3]})
quotes = pd.DataFrame({"item": ["笔", "笔", "笔"], "supplier": ["S1", "S2", "S3"]})
paired = stock.merge(quotes, on="item", how="inner", validate="many_to_many")

print(paired)  # 每个仓库分别匹配 S1、S2、S3。
print(paired.shape)  # (6, 4)，2 × 3 = 6 行。
print(stock["quantity"].sum(), paired["quantity"].sum())  # 5、15。
# 库存总数本来是 5；连接后的 15 包含每个仓库被三个报价重复匹配的数量。

  item warehouse  quantity supplier
0    笔        北仓         2       S1
1    笔        北仓         2       S2
2    笔        北仓         2       S3
3    笔        南仓         3       S1
4    笔        南仓         3       S2
5    笔        南仓         3       S3
(6, 4)
5 15


### 5.2 约束被破坏时停止连接

如果任务约定每个客户只能有一条地区记录，就应使用 many_to_one。右表出现重复客户时，validate 报 MergeError，使问题在结果膨胀前暴露。

改变 validate 为 many_to_many 只会放宽检查，不能修复重复客户。应根据业务决定保留哪条记录或改用更完整的键。

In [7]:
orders = pd.DataFrame({"order": ["O1", "O2"], "customer": ["C1", "C1"]})
customers = pd.DataFrame({"customer": ["C1", "C1"], "region": ["东区", "西区"]})

try:
    orders.merge(customers, on="customer", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print(type(error).__name__)  # MergeError：右侧客户编号不唯一。
else:
    raise AssertionError("预期多对一约束拒绝右侧重复客户")

print(customers)  # 保留两条冲突记录，交由业务规则处理。
print(len(orders), len(customers))  # 两侧原表均仍为 2 行。

MergeError
  customer region
0       C1     东区
1       C1     西区
2 2


## 6 空键会彼此匹配

pandas 的 merge 会把左右键中的缺失值彼此匹配，这不同于通常的 SQL 空值连接行为。空键重复时，也会产生多组配对。

下面左表有一条缺失键，右表有两条缺失键。缺失只表示没有编号，并不能据此断定它们指向同一个实体。

In [8]:
left = pd.DataFrame({"key": ["A", None], "left_record": ["L1", "L2"]})
right = pd.DataFrame({"key": ["A", None, None], "right_record": ["R1", "R2", "R3"]})
matched = left.merge(right, on="key", how="inner", indicator=True)

print(matched)
# A 匹配一行；L2 分别匹配 R2、R3，两条空键配对都标为 both。
print(matched.shape)  # (3, 4)。
print(matched["key"].isna().sum())  # 2：空键不是自动排除的。

   key left_record right_record _merge
0    A          L1           R1   both
1  NaN          L2           R2   both
2  NaN          L2           R3   both
(3, 4)
2


### 6.1 不允许空键匹配时

若业务规定缺少编号的记录不能匹配，可以先把右表空键分离，再进行左连接。左表的空键记录仍保留为未匹配项，右表空键另行输出待核查。

下面沿用刚才的 left、right。这里选择保留左侧所有记录；若采用其他连接方式，空键如何保留也需要单独约定。

In [9]:
right_valid = right.loc[right["key"].notna()]
right_missing = right.loc[right["key"].isna()]
result = left.merge(
    right_valid, on="key", how="left", indicator=True, validate="many_to_one"
)

print(result)
# L1 匹配 R1；L2 仍存在，但 right_record 缺失，来源为 left_only。
print(result.shape, result["_merge"].tolist())  # (2, 4)，['both', 'left_only']。
print(right_missing)  # R2、R3 的原记录没有被无声丢弃。

   key left_record right_record     _merge
0    A          L1           R1       both
1  NaN          L2          NaN  left_only
(2, 4) ['both', 'left_only']
   key right_record
1  NaN           R2
2  NaN           R3


## 7 用 join 按索引连接

DataFrame.join 适合把右表的列按索引连接到左表，默认 how="left"。不传 on 时匹配两侧行标签，也支持 inner、right、outer 等连接方式。

下面两表的索引都是设备编号，顺序不同且只重合一项。用 how 明确保留的设备集合，不能把行位置当作匹配关系。

In [10]:
temperatures = pd.DataFrame({"temperature_c": [20, 18]}, index=["S2", "S1"])
humidity = pd.DataFrame({"humidity_pct": [40, 60]}, index=["S1", "S3"])

for how in ("left", "inner", "outer"):
    result = temperatures.join(humidity, how=how, sort=False, validate="one_to_one")
    print(how, result.index.tolist(), result.shape)
    print(result)
# left 保留 S2、S1；inner 只有 S1；outer 按标签排序为 S1、S2、S3。
# S1 的温度 18 °C 与湿度 40% 配对；其他设备有一侧缺失。

left ['S2', 'S1'] (2, 2)
    temperature_c  humidity_pct
S2             20           NaN
S1             18          40.0
inner ['S1'] (1, 2)
    temperature_c  humidity_pct
S1             18            40
outer ['S1', 'S2', 'S3'] (3, 2)
    temperature_c  humidity_pct
S1           18.0          40.0
S2           20.0           NaN
S3            NaN          60.0


### 7.1 左侧列匹配右侧索引

join(on="customer") 表示用左表 customer 列匹配右表索引，右表不是再按同名数据列匹配。set_index 将客户编号设为右表索引；连接结果保留左侧行标签。

join 的同名非键列用 lsuffix、rsuffix 区分。下面两表无同名非键列，不需要后缀。

In [11]:
orders = pd.DataFrame(
    {"customer": ["C2", "C1"], "quantity": [2, 3]}, index=["O2", "O1"]
)
customers = pd.DataFrame({"customer": ["C1", "C2"], "region": ["东区", "西区"]})
lookup = customers.set_index("customer")
result = orders.join(lookup, on="customer", how="left", validate="many_to_one")

print(result)  # O2 为西区、O1 为东区。
print(result.index.tolist(), result.columns.tolist(), result.shape)
# ['O2', 'O1']，['customer', 'quantity', 'region']，(2, 3)。
print(result.dtypes)  # quantity 为 int64，两列文本为 str。

   customer  quantity region
O2       C2         2     西区
O1       C1         3     东区
['O2', 'O1'] ['customer', 'quantity', 'region'] (2, 3)
customer      str
quantity    int64
region        str
dtype: object


## 8 用 concat 追加记录

concat 沿指定轴拼接对象。axis=0 沿行轴追加，列按列名对齐；join="outer" 取列名并集，join="inner" 只保留共有列。这里的 join 参数处理另一条轴的标签，不是按某个业务列值匹配。

下面两批商品的字段不同。默认保留原行标签；原行号没有业务含义时，可用 ignore_index=True 重新编号，它不会改变列名的对齐规则。

In [12]:
first = pd.DataFrame({"item": ["笔", "本"], "quantity": [2, 3]})
second = pd.DataFrame({"item": ["尺"], "note": ["新到"]})
outer = pd.concat([first, second], axis=0, join="outer", sort=False)
inner = pd.concat([first, second], axis=0, join="inner", ignore_index=True)

print(outer)
# 列顺序为 item、quantity、note；缺少的字段补缺失，quantity 变为 float64。
print(outer.index.tolist(), outer.shape)  # [0, 1, 0]，(3, 3)，没有自动去重。
print(inner)  # 仅有共有的 item 列，依次为笔、本、尺。
print(inner.index.tolist(), inner.shape)  # [0, 1, 2]，(3, 1)。

  item  quantity note
0    笔       2.0  NaN
1    本       3.0  NaN
0    尺       NaN   新到
[0, 1, 0] (3, 3)
  item
0    笔
1    本
2    尺
[0, 1, 2] (3, 1)


### 8.1 横向拼接按行标签对齐

axis=1 沿列轴拼接，行按索引标签对齐。join="outer" 保留行标签并集，join="inner" 只保留共有行标签。

ignore_index=True 只重编号拼接轴：axis=1 时重编号的是列，而不是消除行标签对齐。下面故意让同一设备出现在不同行位置。

In [13]:
left = pd.DataFrame({"temperature_c": [18, 22]}, index=["S1", "S3"])
right = pd.DataFrame({"humidity_pct": [60, 40]}, index=["S3", "S2"])
outer = pd.concat([left, right], axis=1, join="outer", sort=False)
inner = pd.concat([left, right], axis=1, join="inner", sort=False)
renumbered = pd.concat([left, right], axis=1, ignore_index=True, sort=False)

print(outer)  # S3 的 22 °C 与 60% 对齐，S1、S2 各有一侧缺失。
print(outer.index.tolist(), outer.shape)  # ['S1', 'S3', 'S2']，(3, 2)。
print(inner)  # 仅 S3，一行两列。
print(renumbered.index.tolist(), renumbered.columns.tolist())
# 行标签仍为 S1、S3、S2，只把两列重编号为 0、1。

    temperature_c  humidity_pct
S1           18.0           NaN
S3           22.0          60.0
S2            NaN          40.0
['S1', 'S3', 'S2'] (3, 2)
    temperature_c  humidity_pct
S3             22            60
['S1', 'S3', 'S2'] [0, 1]


### 8.2 检查拼接后的重复标签

verify_integrity=True 检查拼接轴的新标签是否重复。它不检查商品编号等数据列是否重复；ignore_index=True 生成的新行号唯一，也不能据此证明业务键唯一。

下面两批记录都使用同一个业务行标签，要求标签唯一时应拒绝这个拼接。

In [14]:
first = pd.DataFrame({"quantity": [2]}, index=["R1"])
second = pd.DataFrame({"quantity": [3]}, index=["R1"])

try:
    pd.concat([first, second], verify_integrity=True)
except ValueError as error:
    print(type(error).__name__)  # ValueError：新行索引中重复出现 R1。
else:
    raise AssertionError("预期完整性检查拒绝重复行标签")

print(first.index.tolist(), second.index.tolist())  # 原两表的 R1 标签都保留。

ValueError
['R1'] ['R1']


## 9 选学：保留批次来源

concat 的 keys 在拼接轴外层添加来源标签，生成 MultiIndex。下面完整行标签由“批次、批次内行号”两部分组成，元组中的顺序与 names 一致。

不同批次内可以有相同局部行号。这里保留这些行号，不同时设置 ignore_index=True；pandas 3 中 keys 与 ignore_index=True 的组合会被拒绝。

In [15]:
first = pd.DataFrame({"quantity": [2, 3]}, index=["R1", "R2"])
second = pd.DataFrame({"quantity": [5]}, index=["R1"])
batches = pd.concat(
    [first, second], keys=["morning", "evening"], names=["batch", "row"],
    verify_integrity=True,
)

print(batches)
print(batches.index.tolist())
# [('morning', 'R1'), ('morning', 'R2'), ('evening', 'R1')]。
print(batches.index.names, batches.shape)  # batch、row 两层，表格为 (3, 1)。
print(batches.index.is_unique)  # True，完整元组标签没有重复。

             quantity
batch   row          
morning R1          2
        R2          3
evening R1          5
[('morning', 'R1'), ('morning', 'R2'), ('evening', 'R1')]
['batch', 'row'] (3, 1)
True


## 10 选学：交叉连接与反连接

### 10.1 交叉连接

how="cross" 把左表每一行与右表每一行配对，不指定 on、left_on、right_on 等连接键。两侧行数分别为 m、n 时，结果有 m × n 行。

下面生成两个颜色与三个尺码的组合。交叉连接前先估计行数，避免把小参数表的写法直接用于两张大表。

In [16]:
colors = pd.DataFrame({"color": ["红", "蓝"]})
sizes = pd.DataFrame({"size": ["S", "M", "L"]})
combinations = colors.merge(sizes, how="cross")

print(combinations)  # 先列红色的 S、M、L，再列蓝色的 S、M、L。
print(len(colors) * len(sizes), combinations.shape)  # 6，(6, 2)。
print(combinations.dtypes)  # 两列均为 str。

  color size
0     红    S
1     红    M
2     红    L
3     蓝    S
4     蓝    M
5     蓝    L
6 (6, 2)
color    str
size     str
dtype: object


### 10.2 找出另一侧没有的键

pandas 3 的 left_anti 保留左表中没有右侧匹配的键，right_anti 则保留右表中没有左侧匹配的键。它们用于查漏，例如找出没有客户资料的订单。

反连接不是去重操作：同一未匹配键有多条记录时仍保留这些记录。空键仍遵循 merge 的匹配规则；下面两侧都有空键，所以空键不在反连接结果中。

In [17]:
left = pd.DataFrame({"key": ["A", "B", "B", None], "left_record": ["L1", "L2", "L3", "L4"]})
right = pd.DataFrame({"key": ["A", "C", None], "right_record": ["R1", "R2", "R3"]})
left_only = left.merge(right, on="key", how="left_anti", sort=False)
right_only = left.merge(right, on="key", how="right_anti", sort=False)

print(left_only[["key", "left_record"]])  # B 的 L2、L3 两行，保留左侧顺序。
print(right_only[["key", "right_record"]])  # C 的 R2 一行。
print(left_only.shape, right_only.shape)  # (2, 3)、(1, 3)，结果仍含两侧字段。
print(left_only["key"].isna().sum(), right_only["key"].isna().sum())  # 0、0。

  key left_record
1   B          L2
2   B          L3


  key right_record
1   C           R2
(2, 3) (1, 3)
0 0


## 11 选学：比较同结构表格

compare 展示两份形状和行列标签相同的表之间的值差异。默认只保留发生差异的行列；相同位置显示为缺失，双方同位置都是缺失时不算差异。

默认横向并列，结果列是“原列名、版本名称”两层标签。keep_shape=True 保留所有原行列位置，keep_equal=True 同时显示相等值；result_names 为两个版本命名。

In [18]:
before = pd.DataFrame(
    {"quantity": [2.0, 4.0, None], "price_yuan": [1.0, 2.0, None]},
    index=["A", "B", "C"],
)
after = before.copy()
after.loc["A", "price_yuan"] = 1.5
after.loc["B", "quantity"] = 5.0
difference = before.compare(after, result_names=("before", "after"))
full = before.compare(after, keep_shape=True, keep_equal=True)

print(difference)
# A 的单价 1.0 → 1.5 元，B 的数量 4 → 5；默认不显示没有变化的 C。
print(difference.index.tolist(), difference.columns.tolist(), difference.shape)
# 两行、四列；列标签由 quantity/price_yuan 与 before/after 组成。
print(full.shape)  # (3, 4)：保留全部行和两份数据的列。

  quantity       price_yuan      
    before after     before after
A      NaN   NaN        1.0   1.5
B      4.0   5.0        NaN   NaN


['A', 'B'] [('quantity', 'before'), ('quantity', 'after'), ('price_yuan', 'before'), ('price_yuan', 'after')] (2, 4)
(3, 4)


### 11.1 compare 不替你按业务键匹配

即使只是行顺序不同，compare 也要求输入的标签顺序一致，否则报 ValueError。它不是把两表按业务键连接后再找差异的工具。

下面先观察拒绝乱序输入，再按明确的原行标签顺序重新选择。真实任务若有新增或删除记录，应先检查键集合，不能为了让 compare 通过而丢弃它们。

In [19]:
before = pd.DataFrame({"quantity": [2, 4]}, index=["A", "B"])
after = pd.DataFrame({"quantity": [5, 2]}, index=["B", "A"])

try:
    before.compare(after)
except ValueError as error:
    print(type(error).__name__)  # ValueError：行标签顺序不同。
else:
    raise AssertionError("预期 compare 拒绝标签顺序不同的输入")

aligned = after.loc[before.index]
print(before.compare(aligned))  # 只有 B：self 为 4，other 为 5。
print(aligned.index.tolist())  # ['A', 'B']，先明确按哪些标签比较。

ValueError
  quantity      
      self other
B      4.0   5.0
['A', 'B']


## 本章小结

（1）merge 按键匹配，how 决定键范围，validate 检查唯一性约束；indicator 用于区分匹配来源，不能只看普通数据列是否缺失。

（2）重复键会产生全部匹配组合，空键也会彼此匹配。连接前后核对键、顺序、行数和统计单位。

（3）join 常用于索引连接；concat 沿一条轴拼接，并按另一条轴的标签对齐。ignore_index 只重编号拼接轴。

（4）keys 保留批次层级，cross 生成全部组合，反连接找出未匹配记录；compare 需要形状和标签一致，不替代按键连接。

## 练习

（1）给每条订单补充地区，保留全部订单和原订单顺序；显式检查客户编号在右表唯一，并用来源列列出没有匹配资料的订单。

In [20]:
orders = pd.DataFrame({"order": ["O3", "O1", "O2"], "customer": ["C2", "C1", "C3"]})
customers = pd.DataFrame({"customer": ["C1", "C2"], "region": ["东区", "西区"]})

# 在此选择 how、validate、indicator 并连接。
# 检查：仍为 3 行，订单顺序为 O3、O1、O2；未匹配订单只有 O2。
# 打印列名、dtype、来源与结果形状。

（2）先预测下面两次拼接的标签、形状和缺失位置，再运行核对。若要求只留下两表都出现的设备，该如何改参数？解释为什么 ignore_index=True 不能取消横向拼接时的行标签对齐。

In [21]:
left = pd.DataFrame({"count": [2, 4]}, index=["S2", "S1"])
right = pd.DataFrame({"value": [1.5, 3.5]}, index=["S1", "S3"])

print(pd.concat([left, right], axis=1, sort=False))
print(pd.concat([left, right], axis=1, ignore_index=True, sort=False))
# 在运行前写下预测；之后打印两表共有设备的拼接结果，并说明参数选择。

    count  value
S2    2.0    NaN
S1    4.0    1.5
S3    NaN    3.5


      0    1
S2  2.0  NaN
S1  4.0  1.5
S3  NaN  3.5


（3）原任务规定每个商品只有一条标准价格，使用 many_to_one 连接。现在右表变成多个供应商报价。先判断原约束是否还成立，再说明新任务是需要所有配对、选择一个报价，还是需要补充连接键；给出你选择的业务约定和理由。

In [22]:
orders = pd.DataFrame({"order": ["O1", "O2"], "item": ["笔", "笔"], "quantity": [2, 3]})
quotes = pd.DataFrame({"item": ["笔", "笔"], "supplier": ["S1", "S2"], "price_yuan": [1.0, 1.5]})

# 先预测直接按 item 匹配会有几行，以及数量之和会怎样变化。
# 在此写出所选业务约定及实现；需要演示失败时捕获具体异常。
# 检查：不要只把 validate 改为 many_to_many，就声称恢复了“每单一个价格”。

（4）两表都有缺失编号，但任务规定缺失编号不能表示同一个设备。保留全部左表记录，并单独列出右表空键记录；完成连接后说明哪些左记录未匹配。

In [23]:
left = pd.DataFrame({"device": ["S1", None, "S3"], "reading": [10, 20, 30]})
right = pd.DataFrame({"device": ["S1", None], "region": ["东区", "待确认"]})

# 在此分离右表空键，按规则左连接并查看来源。
# 检查：结果仍为 3 行，只有 S1 匹配；左侧空键与 S3 都未匹配。
# 打印右表待核查记录，避免无声丢弃。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) 的 how、on、left_on/right_on、sort、suffixes、indicator、validate 与空键 Warning：连接范围、顺序、键、来源及基数约束；[Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html) 的 Merge types、Merge key uniqueness、Joining logic of the resulting axis：多对多笛卡尔配对、唯一性与标签对齐；[DataFrame.join](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) 的 on、how、lsuffix/rsuffix、validate：索引及列对索引连接；[concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) 的 axis、join、ignore_index、keys、names、verify_integrity、sort：拼接轴、另一轴的标签对齐及层级标签；[DataFrame.compare](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.compare.html) 的 Parameters、Raises、Notes、Examples：差异展示、标签条件和相同 NaN；[pandas 3.0.0 release notes](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html) 的 Other enhancements / Reshaping：反连接与 keys/ignore_index 的限制；[Series.is_unique](https://pandas.pydata.org/docs/reference/api/pandas.Series.is_unique.html) 的唯一性判断；[set_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html) 的列转索引；[String dtype migration](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html) 的 Background、Brief introduction：默认 str 与当前 PyArrow 存储；[Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) 的 Values considered missing、isna/notna：缺失检测及数值 dtype 变化。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[merging](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/merging.rst)、[v3.0.0](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/whatsnew/v3.0.0.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)、[missing_data](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/missing_data.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |